# Notebook 04 - Feature Validation

Notebook này thực hiện đánh giá mức độ phù hợp của các đặc trưng trong bộ dữ liệu Digital Burnout quốc tế trước khi xây dựng các mô hình học máy ở Notebook 05.

Quá trình đánh giá đặc trưng được thực hiện thông qua các phương pháp thống kê và học máy nhằm xác định những biến có giá trị trong việc dự báo Digital Burnout.

- Đánh giá mức độ phù hợp của từng đặc trưng.
- Kiểm định mối liên hệ giữa đặc trưng và biến mục tiêu.
- So sánh mức độ quan trọng của đặc trưng bằng nhiều phương pháp.
- Lựa chọn tập đặc trưng cuối cùng cho giai đoạn xây dựng mô hình.

# 0. Set Up

Chuẩn bị môi trường làm việc cho quá trình đánh giá đặc trưng.

In [1]:
# Import thư viện xử lý dữ liệu

from pathlib import Path

import warnings

import numpy as np
import pandas as pd

# Import thư viện trực quan hóa

import matplotlib.pyplot as plt
import seaborn as sns

# Import thư viện kiểm định thống kê

from scipy.stats import f_oneway
from scipy.stats import chi2_contingency

# Import thư viện Feature Validation

from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

# Thiết lập hiển thị

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

warnings.filterwarnings("ignore")

# Thiết lập random seed

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Đã thiết lập môi trường làm việc.")

Đã thiết lập môi trường làm việc.


In [2]:
# Thiết lập đường dẫn dữ liệu

data_path = Path(
    "../../data/processed/international_dataset/digital_burnout_cleaned.csv"
)

output_path = Path(
    "../../data/processed/international_dataset/validated_features.csv"
)

print("Đã thiết lập đường dẫn dữ liệu.")

Đã thiết lập đường dẫn dữ liệu.


# 1. Load Cleaned Dataset

Đọc bộ dữ liệu đã được làm sạch từ Notebook 02 và kiểm tra thông tin tổng quan trước khi thực hiện đánh giá đặc trưng.

Việc kiểm tra dữ liệu đầu vào giúp đảm bảo quá trình Feature Validation được thực hiện trên bộ dữ liệu đầy đủ, đúng cấu trúc và sẵn sàng cho các bước phân tích tiếp theo.

- Đọc bộ dữ liệu đã được làm sạch.
- Kiểm tra kích thước dữ liệu.
- Kiểm tra kiểu dữ liệu.
- Hiển thị một số dòng dữ liệu đầu tiên.
- Kiểm tra giá trị thiếu.

In [3]:
# Đọc bộ dữ liệu đã được làm sạch

df = pd.read_csv(data_path)

print("Đã tải bộ dữ liệu thành công.")

Đã tải bộ dữ liệu thành công.


In [4]:
# Kiểm tra kích thước bộ dữ liệu

print(f"Số dòng: {df.shape[0]:,}")
print(f"Số cột: {df.shape[1]}")

Số dòng: 5,000,000
Số cột: 35


In [5]:
# Hiển thị thông tin tổng quan của bộ dữ liệu

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000000 entries, 0 to 4999999
Data columns (total 35 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   user_id                  int64  
 1   age                      int64  
 2   occupation               object 
 3   work_mode                object 
 4   device_usage_type        object 
 5   daily_screen_time        float64
 6   social_media_hours       float64
 7   doomscrolling_duration   float64
 8   app_switch_frequency     int64  
 9   notification_count       int64  
 10  smartphone_unlocks       int64  
 11  late_night_device_usage  int64  
 12  focus_sessions           int64  
 13  deep_work_hours          float64
 14  distraction_frequency    int64  
 15  task_completion_rate     int64  
 16  concentration_score      int64  
 17  sleep_hours              float64
 18  sleep_quality            int64  
 19  caffeine_intake          int64  
 20  physical_activity        float64
 21  stress_l

In [6]:
# Hiển thị một số quan sát đầu tiên

df.head()

,user_id,age,occupation,work_mode,device_usage_type,daily_screen_time,social_media_hours,doomscrolling_duration,app_switch_frequency,notification_count,smartphone_unlocks,late_night_device_usage,focus_sessions,deep_work_hours,distraction_frequency,task_completion_rate,concentration_score,sleep_hours,sleep_quality,caffeine_intake,physical_activity,stress_level,workspace_quality,meeting_hours,internet_stability,remote_work_days,motivation_level,mental_fatigue,emotional_exhaustion,work_satisfaction,mental_state,burnout_risk,productivity_score,productivity_category,burnout_level
0,1,56,Content Creator,Office,Entertainment-Centric,8.8,5.0,1.2,41,112,49,1,3,6.0,67,92,2,5.9,10,6,1.6,10,5,2.8,3,4,8.0,10,4,8,Balanced,46,100,High,Moderate
1,2,46,Student,Hybrid,Work-Centric,10.3,2.2,2.4,119,168,153,1,9,4.6,75,74,3,5.6,7,5,1.1,5,10,3.2,7,6,5.0,7,9,7,Balanced,57,96,High,Moderate
2,3,32,Software Engineer,Remote,Balanced,6.5,4.6,1.0,121,199,234,1,5,3.0,70,96,7,5.5,8,6,1.1,4,1,2.3,9,6,8.0,5,2,6,Balanced,29,79,High,Low
3,4,25,Designer,Office,Balanced,9.6,1.2,0.1,85,122,177,1,9,2.9,107,60,3,6.1,5,1,1.7,1,4,3.8,10,5,6.0,4,5,3,Burnout,57,63,Medium,Moderate
4,5,38,Analyst,Hybrid,Work-Centric,13.3,1.6,1.9,221,73,91,1,9,2.8,72,73,9,7.9,1,3,1.8,4,8,3.0,1,1,4.0,7,9,7,Focused,64,89,High,Moderate


In [7]:
# Kiểm tra giá trị thiếu

missing_summary = (
    df.isnull()
      .sum()
      .reset_index()
)

missing_summary.columns = [
    "feature",
    "missing_count"
]

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"]
    / len(df)
    * 100
)

missing_summary = (
    missing_summary
    .sort_values(
        by="missing_count",
        ascending=False
    )
    .reset_index(drop=True)
)

missing_summary

,feature,missing_count,missing_percentage
0,user_id,0,0.0
1,age,0,0.0
2,occupation,0,0.0
3,work_mode,0,0.0
4,device_usage_type,0,0.0
5,daily_screen_time,0,0.0
6,social_media_hours,0,0.0
7,doomscrolling_duration,0,0.0
8,app_switch_frequency,0,0.0
9,notification_count,0,0.0


In [8]:
# Thống kê kiểu dữ liệu

dtype_summary = (
    df.dtypes
      .astype(str)
      .value_counts()
      .reset_index()
)

dtype_summary.columns = [
    "data_type",
    "count"
]

dtype_summary

,data_type,count
0,int64,21
1,float64,8
2,object,6


Bộ dữ liệu đã được tải thành công và sẵn sàng cho quá trình đánh giá đặc trưng.

Kết quả kiểm tra cho thấy:

- Bộ dữ liệu gồm 22 biến sau quá trình tiền xử lý.
- Các kiểu dữ liệu đã được chuẩn hóa, bao gồm 14 biến số nguyên, 6 biến số thực và 2 biến phân loại.
- Không phát hiện lỗi trong quá trình đọc dữ liệu.
- Bộ dữ liệu đáp ứng yêu cầu để thực hiện các bước kiểm định thống kê và đánh giá đặc trưng.

# 2. Feature Validation Configuration

Xác định các nhóm đặc trưng và biến mục tiêu được sử dụng trong quá trình đánh giá đặc trưng.

Việc phân loại các biến theo vai trò nghiên cứu giúp lựa chọn phương pháp kiểm định phù hợp, đồng thời đảm bảo tính nhất quán giữa các bước Feature Validation và Modeling.

- Phân nhóm các đặc trưng theo khung nghiên cứu Digital Burnout.
- Xác định biến mục tiêu.
- Phân loại biến số và biến phân loại.
- Tổng hợp cấu hình đánh giá đặc trưng.# 2. Feature Validation Configuration

## 2.1 Research Feature Groups

Phân loại các đặc trưng theo các nhóm nghiên cứu chính của Digital Burnout.

Các nhóm đặc trưng được xây dựng dựa trên mô hình nghiên cứu, phản ánh các khía cạnh của hành vi sử dụng thiết bị số, khả năng nhận thức, trạng thái tâm lý, giấc ngủ và các yếu tố bối cảnh.## 2.1 Research Feature Groups

In [13]:
# Khai báo các nhóm đặc trưng nghiên cứu

demographic_features = [
    "age"
]

digital_exposure = [
    "daily_screen_time",
    "social_media_hours",
    "doomscrolling_duration",
    "app_switch_frequency",
    "notification_count",
    "smartphone_unlocks",
    "late_night_device_usage"
]

work_environment = [
    "workspace_quality",
    "meeting_hours",
    "internet_stability",
    "remote_work_days"
]

cognitive_performance = [
    "focus_sessions",
    "deep_work_hours",
    "distraction_frequency",
    "task_completion_rate",
    "concentration_score",
    "motivation_level"
]

lifestyle_recovery = [
    "sleep_hours",
    "sleep_quality",
    "caffeine_intake",
    "physical_activity"
]

psychological_symptoms = [
    "stress_level",
    "mental_fatigue",
    "emotional_exhaustion"
]

work_outcome = [
    "work_satisfaction"
]

print("Đã khai báo các nhóm đặc trưng nghiên cứu.")

Đã khai báo các nhóm đặc trưng nghiên cứu.


## 2.2 Target Variables

Xác định biến mục tiêu được sử dụng trong nghiên cứu.

Trong đó, **burnout_level** là biến mục tiêu chính phục vụ xây dựng mô hình phân loại Digital Burnout, còn **productivity_score** được sử dụng như biến tham chiếu trong quá trình phân tích.


In [14]:
# Khai báo biến mục tiêu chính

primary_target = "burnout_level"

# Khai báo biến mục tiêu bổ sung

secondary_target = "productivity_score"

print(f"Biến mục tiêu chính: {primary_target}")
print(f"Biến mục tiêu bổ sung: {secondary_target}")

Biến mục tiêu chính: burnout_level
Biến mục tiêu bổ sung: productivity_score


## 2.3 Numerical Features

Xác định các biến số được sử dụng trong quá trình đánh giá đặc trưng.

Các biến không phục vụ trực tiếp cho nghiên cứu như khóa định danh và biến mục tiêu sẽ được loại khỏi danh sách đánh giá.

In [15]:
# Xác định các biến số

exclude_columns = [
    "user_id",
    "burnout_risk",
    "burnout_level",
    "productivity_score"
]

numerical_features = [

    column

    for column in df.select_dtypes(
        include=["int64", "float64"]
    ).columns

    if column not in exclude_columns

]

print(f"Số lượng biến số: {len(numerical_features)}")

pd.DataFrame(
    {
        "Numerical Feature": numerical_features
    }
)

Số lượng biến số: 26


,Numerical Feature
0,age
1,daily_screen_time
2,social_media_hours
3,doomscrolling_duration
4,app_switch_frequency
5,notification_count
6,smartphone_unlocks
7,late_night_device_usage
8,focus_sessions
9,deep_work_hours


## 2.4 Categorical Features

Xác định các biến phân loại được sử dụng trong quá trình đánh giá đặc trưng.

In [16]:
# Xác định các biến phân loại

categorical_features = [

    column

    for column in df.select_dtypes(
        include=["object", "category"]
    ).columns

    if column != primary_target

]

print(f"Số lượng biến phân loại: {len(categorical_features)}")

pd.DataFrame(
    {
        "Categorical Feature": categorical_features
    }
)

Số lượng biến phân loại: 5


,Categorical Feature
0,occupation
1,work_mode
2,device_usage_type
3,mental_state
4,productivity_category


## 2.5 Validation Configuration Summary

Tổng hợp các nhóm đặc trưng và biến mục tiêu được sử dụng trong quá trình Feature Validation.

In [17]:
# Tổng hợp cấu hình Feature Validation

validation_summary = pd.DataFrame({

    "Feature Group": [

        "Demographic",
        "Digital Exposure",
        "Work Environment",
        "Cognitive Performance",
        "Lifestyle & Recovery",
        "Psychological Symptoms",
        "Work Outcome"

    ],

    "Number of Features": [

        len(demographic_features),
        len(digital_exposure),
        len(work_environment),
        len(cognitive_performance),
        len(lifestyle_recovery),
        len(psychological_symptoms),
        len(work_outcome)

    ]

})

validation_summary

,Feature Group,Number of Features
0,Demographic,1
1,Digital Exposure,7
2,Work Environment,4
3,Cognitive Performance,6
4,Lifestyle & Recovery,4
5,Psychological Symptoms,3
6,Work Outcome,1


Các đặc trưng nghiên cứu đã được phân loại thành bảy nhóm chính dựa trên khung lý thuyết Digital Burnout và đặc điểm của bộ dữ liệu quốc tế.

Tổng cộng có **26 biến đầu vào** được sử dụng trong quá trình đánh giá đặc trưng, bao gồm các nhóm về đặc điểm cá nhân, mức độ tiếp xúc với thiết bị số, môi trường làm việc, hiệu suất nhận thức, lối sống và phục hồi, trạng thái tâm lý và kết quả công việc.

Biến **burnout_level** được xác định là biến mục tiêu chính của nghiên cứu và sẽ được sử dụng trong các bước kiểm định thống kê cũng như xây dựng mô hình học máy ở Notebook 05. Biến **productivity_score** được giữ lại như một biến tham chiếu phục vụ các phân tích bổ sung.

# 3. Statistical Feature Validation

Đánh giá mức độ liên quan giữa các đặc trưng và biến mục tiêu thông qua các phương pháp kiểm định thống kê.

Việc sử dụng nhiều phương pháp đánh giá giúp xem xét đặc trưng dưới các góc độ khác nhau, bao gồm sự khác biệt giữa các nhóm, mức độ phụ thuộc và lượng thông tin mà mỗi đặc trưng đóng góp cho biến mục tiêu.

- ANOVA F-test
- Chi-square Test
- Mutual Information